In [1]:
%pip install azure-eventhub faker

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 7, Finished, Available, Finished, True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 5.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



### Imports & Config

In [2]:
import json
import random
import asyncio
from datetime import datetime, date
from faker import Faker
from azure.eventhub.aio import EventHubProducerClient
from azure.eventhub import EventData

fake = Faker()


CONNECTION_STR = "Endpoint=sb://esehsec1udzbw197hovt1cv.servicebus.windows.net/;SharedAccessKeyName=key_0f9896ef-c4a2-449c-82c9-e1b77641ed02;SharedAccessKey=eNCOg4WBRFUuHr+PZ3GYyCc0Cl2VRCe9W+AEhAeqGQk=;EntityPath=es_9efb865f-8ea3-4430-b25f-5712d525f20c"
EVENT_HUB_NAME = "es_9efb865f-8ea3-4430-b25f-5712d525f20c"


TODAY = date.today()
TODAY_STR = TODAY.strftime("%Y-%m-%d")


NUM_EXISTING_CUSTOMER_ORDERS = 500
NUM_NEW_CUSTOMER_ORDERS = 100
NUM_CUSTOMER_UPDATES = 30
NUM_PRICE_CHANGES = 3


VALID_PAYMENT_METHODS = ["Credit Card", "PayPal", "Cash", "Bank Transfer", "Debit Card"]
VALID_GENDERS = ["Male", "Female", "Other"]

print(f" Event date: {TODAY_STR}")
print(" Config loaded")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 9, Finished, Available, Finished, False)

 Event date: 2026-04-26
 Config loaded


### Load Reference Data from Gold

In [3]:
from pyspark.sql import functions as F

GOLD_NAME = "Gold_Lakehouse"
GOLD_PATH = f"/lakehouse/added/{GOLD_NAME}/Tables"

# --- Existing customers ---
df_customers = spark.read.format("delta").load("abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/da923130-af44-4b41-9efd-b133ac83e6b7/Tables/dbo/dim_customer")
existing_customer_ids = [row.customer_id for row in 
    df_customers.filter("customer_id != -1").select("customer_id").collect()
]
max_customer_id = max(existing_customer_ids)

# --- Existing products ---
df_products = spark.read.format("delta").load("abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/da923130-af44-4b41-9efd-b133ac83e6b7/Tables/dbo/dim_product")
existing_products = [{
    "product_id": row.product_id,
    "price": float(row.price) if row.price else 0.0
} for row in 
    df_products.filter("product_id != -1").select("product_id", "price").collect()
]

# --- Max IDs ---
df_sales = spark.read.format("delta").load("abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/da923130-af44-4b41-9efd-b133ac83e6b7/Tables/dbo/fact_sales")
max_order_id = df_sales.agg(F.max("order_id")).collect()[0][0]
max_order_item_id = df_sales.agg(F.max("order_item_id")).collect()[0][0]

print(f"Customers: {len(existing_customer_ids):,} (max: {max_customer_id})")
print(f"Products:  {len(existing_products):,}")
print(f"Max order ID: {max_order_id:,}")
print(f"Max order item ID: {max_order_item_id:,}")
print("Reference data loaded")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 10, Finished, Available, Finished, False)

Customers: 1,048,724 (max: 1048774)
Products:  20,000
Max order ID: 8,001,198
Max order item ID: 20,003,595
Reference data loaded


###  ID Counter + Unified Event Wrapper

In [4]:
# AUTO-INCREMENT IDS
id_counters = {
    "order_id": max_order_id,
    "order_item_id": max_order_item_id,
    "customer_id": max_customer_id
}

def next_id(key):
    id_counters[key] += 1
    return id_counters[key]


# UNIFIED EVENT WRAPPER
# Every event gets the SAME 3 columns:
#   event_type (string)
#   timestamp  (string)
#   payload    (JSON string) ← ALL data goes here
#
# This way Eventstream never drops fields!
def wrap_event(event_type, data):
    return {
        "event_type": event_type,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "payload": json.dumps(data, default=str)
    }


print("ID counters & event wrapper loaded")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 11, Finished, Available, Finished, False)

ID counters & event wrapper loaded


### Event Generator Functions

In [5]:
# EVENT 1: New order from EXISTING customer
def generate_existing_customer_order():
    customer_id = random.choice(existing_customer_ids)
    order_id = next_id("order_id")
    
    items = []
    for _ in range(random.randint(1, 5)):
        product = random.choice(existing_products)
        quantity = random.randint(1, 5)
        unit_price = round(product["price"] * random.uniform(0.9, 1.1), 2)
        items.append({
            "order_item_id": next_id("order_item_id"),
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": quantity,
            "unit_price": unit_price
        })
    
    total_amount = round(sum(i["quantity"] * i["unit_price"] for i in items), 2)
    
    return wrap_event("existing_customer_order", {
        "order": {
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": TODAY_STR,
            "total_amount": total_amount,
            "payment_method": random.choice(VALID_PAYMENT_METHODS),
            "shipping_country": fake.country()
        },
        "order_items": items
    })


# EVENT 2: New order from NEW customer
def generate_new_customer_order():
    customer_id = next_id("customer_id")
    order_id = next_id("order_id")
    
    items = []
    for _ in range(random.randint(1, 5)):
        product = random.choice(existing_products)
        quantity = random.randint(1, 5)
        unit_price = round(product["price"] * random.uniform(0.9, 1.1), 2)
        items.append({
            "order_item_id": next_id("order_item_id"),
            "order_id": order_id,
            "product_id": product["product_id"],
            "quantity": quantity,
            "unit_price": unit_price
        })
    
    total_amount = round(sum(i["quantity"] * i["unit_price"] for i in items), 2)
    
    return wrap_event("new_customer_order", {
        "customer": {
            "customer_id": customer_id,
            "name": fake.name(),
            "email": fake.email(),
            "gender": random.choice(VALID_GENDERS),
            "start_date": TODAY_STR,
            "country": fake.country(),
            "end_date": None,
            "status": 1
        },
        "order": {
            "order_id": order_id,
            "customer_id": customer_id,
            "order_date": TODAY_STR,
            "total_amount": total_amount,
            "payment_method": random.choice(VALID_PAYMENT_METHODS),
            "shipping_country": fake.country()
        },
        "order_items": items
    })

# EVENT 3: Existing customer data changed
def generate_customer_update():
    customer_id = random.choice(existing_customer_ids)
    change_type = random.choices(
        ["email_change", "country_change"],
        weights=[0.20, 0.80], k=1
    )[0]
    
    data = {
        "customer_id": customer_id,
        "change_type": change_type,
        "change_date": TODAY_STR
    }
    
    if change_type == "email_change":
        data["new_email"] = fake.email()
    elif change_type == "country_change":
        data["new_country"] = fake.country()

    
    return wrap_event("customer_update", data)


# EVENT 4: Product price change
def generate_price_change():
    product = random.choice(existing_products)
    old_price = product["price"]
    change_pct = random.uniform(-0.20, 0.30)
    new_price = max(round(old_price * (1 + change_pct), 2), 1.00)
    
    return wrap_event("price_change", {
        "product_id": product["product_id"],
        "old_price": old_price,
        "new_price": new_price,
        "price_change_pct": round(change_pct * 100, 2),
        "effective_date": TODAY_STR,
        "reason": random.choice([
            "Market adjustment", "Competitor pricing",
            "Cost increase", "Promotional pricing",
            "Seasonal adjustment", "Supplier cost change"
        ])
    })


print(" Event generators loaded (unified schema)")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 12, Finished, Available, Finished, False)

 Event generators loaded (unified schema)


### Generate All Events

In [6]:
print("=" * 60)
print(f"🎲 GENERATING EVENTS FOR {TODAY_STR}")
print("=" * 60)

all_events = []

for _ in range(NUM_EXISTING_CUSTOMER_ORDERS):
    all_events.append(generate_existing_customer_order())
print(f"    Existing customer orders: {NUM_EXISTING_CUSTOMER_ORDERS}")

for _ in range(NUM_NEW_CUSTOMER_ORDERS):
    all_events.append(generate_new_customer_order())
print(f"    New customer orders:      {NUM_NEW_CUSTOMER_ORDERS}")

for _ in range(NUM_CUSTOMER_UPDATES):
    all_events.append(generate_customer_update())
print(f"    Customer updates:         {NUM_CUSTOMER_UPDATES}")

for _ in range(NUM_PRICE_CHANGES):
    all_events.append(generate_price_change())
print(f"    Price changes:            {NUM_PRICE_CHANGES}")

random.shuffle(all_events)
print(f"\n  Total events: {len(all_events)}")

# --- Verify unified schema ---
print(f"\n  Schema check — every event has same keys:")
for et in ["existing_customer_order", "new_customer_order", "customer_update", "price_change"]:
    sample = next(e for e in all_events if e["event_type"] == et)
    print(f"      {et}: keys = {list(sample.keys())}")
    print(f"         payload preview: {sample['payload'][:100]}...")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 13, Finished, Available, Finished, False)

🎲 GENERATING EVENTS FOR 2026-04-26
    Existing customer orders: 500
    New customer orders:      100
    Customer updates:         30
    Price changes:            3

  Total events: 633

  Schema check — every event has same keys:
      existing_customer_order: keys = ['event_type', 'timestamp', 'payload']
         payload preview: {"order": {"order_id": 8001463, "customer_id": 687721, "order_date": "2026-04-26", "total_amount": 1...
      new_customer_order: keys = ['event_type', 'timestamp', 'payload']
         payload preview: {"customer": {"customer_id": 1048797, "name": "Jared Jones", "email": "michellebailey@example.org", ...
      customer_update: keys = ['event_type', 'timestamp', 'payload']
         payload preview: {"customer_id": 300418, "change_type": "country_change", "change_date": "2026-04-26", "new_country":...
      price_change: keys = ['event_type', 'timestamp', 'payload']
         payload preview: {"product_id": 6429, "old_price": 229.76, "new_price": 274.15, "pr

### Send Events to Eventstream

In [7]:
async def send_events_to_eventstream(events, batch_size=50):
    producer = EventHubProducerClient.from_connection_string(
        conn_str=CONNECTION_STR,
        eventhub_name=EVENT_HUB_NAME
    )
    
    total_batches = 0
    
    async with producer:
        for i in range(0, len(events), batch_size):
            batch_events = events[i:i + batch_size]
            event_data_batch = await producer.create_batch()
            
            for event in batch_events:
                event_json = json.dumps(event, default=str)
                try:
                    event_data_batch.add(EventData(event_json))
                except ValueError:
                    await producer.send_batch(event_data_batch)
                    total_batches += 1
                    event_data_batch = await producer.create_batch()
                    event_data_batch.add(EventData(event_json))
            
            await producer.send_batch(event_data_batch)
            total_batches += 1
            
            sent = min(i + batch_size, len(events))
            print(f" Sent {sent}/{len(events)} events...")
    
    return total_batches

print("=" * 60)
print("SENDING EVENTS TO EVENTSTREAM")
print("=" * 60)

total_batches = await send_events_to_eventstream(all_events)
print(f"\n  All {len(all_events)} events sent in {total_batches} batches!")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 14, Finished, Available, Finished, False)

SENDING EVENTS TO EVENTSTREAM
 Sent 50/633 events...
 Sent 100/633 events...
 Sent 150/633 events...
 Sent 200/633 events...
 Sent 250/633 events...
 Sent 300/633 events...
 Sent 350/633 events...
 Sent 400/633 events...
 Sent 450/633 events...
 Sent 500/633 events...
 Sent 550/633 events...
 Sent 600/633 events...
 Sent 633/633 events...

  All 633 events sent in 13 batches!


### Daily Summary Report

In [8]:
total_revenue = sum(
    json.loads(e["payload"])["order"]["total_amount"]
    for e in all_events 
    if e["event_type"] in ["existing_customer_order", "new_customer_order"]
)

price_events = [e for e in all_events if e["event_type"] == "price_change"]

print("\n")
print("╔" + "═" * 60 + "╗")
print("║" + f"  EVENT SUMMARY — {TODAY_STR}".center(60) + "║")
print("╠" + "═" * 60 + "╣")
print(f"║   Existing customer orders:  {NUM_EXISTING_CUSTOMER_ORDERS:>6}                     ║")
print(f"║   New customer orders:       {NUM_NEW_CUSTOMER_ORDERS:>6}                          ║")
print(f"║   Customer updates:          {NUM_CUSTOMER_UPDATES:>6}                             ║")
print(f"║   Price changes:             {NUM_PRICE_CHANGES:>6}                                ║")
print(f"║   Total events:              {len(all_events):>6}                                  ║")
print(f"║   Total revenue:       ${total_revenue:>12,.2f}                                    ║")
print(f"║                                                                                    ║")
print(f"║   Schema: event_type + timestamp + payload (unified)                               ║")
print("╚" + "═" * 60 + "╝")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 15, Finished, Available, Finished, False)



╔════════════════════════════════════════════════════════════╗
║                  EVENT SUMMARY — 2026-04-26                ║
╠════════════════════════════════════════════════════════════╣
║   Existing customer orders:     500                     ║
║   New customer orders:          100                     ║
║   Customer updates:              30                     ║
║   Price changes:                  3                     ║
║   Total events:                 633                     ║
║   Total revenue:       $1,314,830.83                ║
║                                                            ║
║   Schema: event_type + timestamp + payload (unified)     ║
╚════════════════════════════════════════════════════════════╝


### Verify Data Landed in Bronze

In [9]:
import time
print("⏳ Waiting 15 seconds for Eventstream...")
time.sleep(15)

BRONZE_NAME = "Bronze_Layer"

try:
    df = spark.read.format("delta").load(
        "abfss://7957e7ae-0f8c-4c1e-bd4d-034069053e54@onelake.dfs.fabric.microsoft.com/2f855148-2e3c-4ca1-b8a3-6d49d05e642d/Tables/dbo/streaming_events_v3"
    )
    print(f" Events landed! {df.count():,} rows")
    print(f" Columns: {df.columns}")
    df.printSchema()
    
    for et in df.select("event_type").distinct().collect():
        print(f"\n {et[0]}:")
        df.filter(F.col("event_type") == et[0]).show(2, truncate=80)

except Exception as e:
    print(f" {str(e)[:200]}")

StatementMeta(, 1d7e0b86-5a5b-42c6-962d-67bc986470db, 16, Finished, Available, Finished, False)

⏳ Waiting 15 seconds for Eventstream...
 Events landed! 1,566 rows
 Columns: ['event_type', 'timestamp', 'payload']
root
 |-- event_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- payload: string (nullable = true)


 existing_customer_order:
+-----------------------+---------------------------+--------------------------------------------------------------------------------+
|             event_type|                  timestamp|                                                                         payload|
+-----------------------+---------------------------+--------------------------------------------------------------------------------+
|existing_customer_order|2026-04-24T14:46:57.769684Z|{"order": {"order_id": 8000099, "customer_id": 261177, "order_date": "2026-04...|
|existing_customer_order|2026-04-24T14:46:57.765853Z|{"order": {"order_id": 8000013, "customer_id": 132927, "order_date": "2026-04...|
+-----------------------+---------------------------+-